In [3]:
import sys
sys.path.insert(0,'/mnt/AEA8F340A8F3059D/sportsbet/ai-engine')

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from tools.mongodbtools import fetchMatch,fetchMatchOdds,fetchOddsHistory
from tools.oddstools import calculateEv,kellyCriterion,oddsToProbability,oddsMovementAnalysis
from config.settings import settings
import json,logging

In [5]:
logger=logging.getLogger(__name__)

In [6]:
BET_ADVISOR_PROMPT="""You are an expert sports betting advisor with deep knowledge of cricket and probability.

MATCH DATA:
{match_data}

CURRENT ODDS & OUTCOMES:
{odds_data}

ODDS MOVEMENT ANALYSIS:
{odds_movement}

EV CALCULATIONS:
{ev_data}

KELLY CRITERION SUGGESTIONS:
{kelly_data}

USER'S WALLET & RECENT BETS:
{user_context}

RAG CONTEXT:
{rag_context}

Provide detailed betting advice as JSON:
{{
  "recommendation": "bet|hold|avoid",
  "summary": "2-3 sentence summary of advice",
  "evAnalysis": {{
    "bestBet": "which outcome has best EV",
    "ev": 0.0,
    "explanation": "why this bet has positive EV"
  }},
  "kellySuggestion": {{
    "recommendedStake": 0.0,
    "fractionOfBankroll": 0.0,
    "reasoning": "why this stake size"
  }},
  "oddsMovement": {{
    "trend": "shortening|drifting|stable",
    "insight": "what the odds movement tells us"
  }},
  "riskAssessment": {{
    "level": "low|medium|high",
    "factors": ["list of risk factors"],
    "mitigations": ["how to reduce risk"]
  }},
  "confidence": 0.0-1.0
}}
"""

In [9]:
async def betAdvisor(state):
    ctx=state.get("context",{})
    matchId=ctx.get("matchId","") or state.get("slots",{}).get("matchId","")
    matchData=state.get("match-data",{})
    if not matchData and matchId:
        matchData=await fetchMatch(matchId) or {}
    oddsData=await fetchMatchOdds(matchId) if matchId else {}
    outcomes=oddsData.get("_outcomes",[]) if oddsData else []

    oddsHistory=[]
    if oddsData:
        oddsHistory=await fetchOddsHistory(str(oddsData.get("_id","")))
    oddsMovement=oddsMovementAnalysis(oddsHistory)

    evResults=[]
    kellyResults=[]
    walletBalance=state.get("user-wallet",{}).get("balance",1000)

    for outcome in outcomes:
        odds=outcome.get("odds",2.0)
        probability=oddsToProbability(odds)
        ev=calculateEv(odds,probability)
        kelly=KellyCriterion(odds,probability,walletBalance)
        evResults.append({
            "team":outcome.get("teamname",""),
            "odds":odds,
            "probability":probability,
            "ev":ev
        })
        kellyResults.append({
            "team":outcome.get("teamname",""),
            "kelly":kelly
        })
    llm=ChatGoogleGenerativeAI(
        model=settings.GEMINI_MODEL,
        google_api_key=settings.GEMINI_API_KEY,
        temperature=0.2
    )
    prompt=ChatPromptTemplate.from_messages([
        ("system",BET_ADVISOR_PROMPT)
    ])
    try:
        result=await (prompt|llm).ainvoke({
            "match-data":json.dumps(matchData,default=str)[:1000],
            "odds-data":json.dumps(oddsData,default=str)[:1000],
            "odds-movement":json.dumps(oddsMovement,default=str)[:1000],
            "ev-data":json.dumps(evResults,default=str)[:1000],
            "kelly-data":json.dumps(kellyResults,default=str)[:1000],
            "user-context":json.dumps({
                "wallet":state.get("user-wallet",{}),
                "recentBets":state.get("user-recent-bets",[])
            },default=str)[:1000],
            "rag-context":json.dumps(state.get("rag-context",{}),default=str)[:1000]
        })
        content=result.content.strip()
        if "```" in content:
            content=content.split("```")[1].split("```")[0]
            if content.startswith("json"):
                content=content[4:]
        output=json.loads(content)
    except Exception as e:
        logger.error(f"bet advisor failed {e}")
        output={
            "recommendation":"hold",
            "error":str(e)
        }
    return{
        "output":output,
        "match-data":matchData,
    }